# Digital DNA — 03: Model Training & Tuning
Fitting One-Class SVM and Isolation Forest models on genuine enrollment samples and evaluating anomaly detection decision boundaries.

In [ ]:
import sys
from pathlib import Path
import numpy as np
import pandas as pd

project_root = Path("..").resolve()
if str(project_root) not in sys.path:
    sys.path.insert(0, str(project_root))

from features.feature_pipeline import FeaturePipeline
from models.keystroke_model import KeystrokeModel
from models.mouse_model import MouseModel
from models.model_utils import load_model

## 1. Train Keystroke Model (One-Class SVM)

In [ ]:
gen_k = pd.read_csv(project_root / "data" / "processed" / "genuine_keystroke.csv").drop(columns=["label"], errors="ignore")
imp_k = pd.read_csv(project_root / "data" / "processed" / "impostor_keystroke.csv").drop(columns=["label"], errors="ignore")

# Fit model on genuine data
k_model = KeystrokeModel(nu=0.05, kernel="rbf", gamma="scale").fit(gen_k.values)

gen_scores = [k_model.score(x) for x in gen_k.values]
imp_scores = [k_model.score(x) for x in imp_k.values]

print(f"Genuine Keystroke Mean Similarity:  {np.mean(gen_scores):.3f} (std: {np.std(gen_scores):.3f})")
print(f"Impostor Keystroke Mean Similarity: {np.mean(imp_scores):.3f} (std: {np.std(imp_scores):.3f})")

## 2. Train Mouse Model (Isolation Forest)

In [ ]:
gen_m = pd.read_csv(project_root / "data" / "processed" / "genuine_mouse.csv").drop(columns=["label"], errors="ignore")
imp_m = pd.read_csv(project_root / "data" / "processed" / "impostor_mouse.csv").drop(columns=["label"], errors="ignore")

# Fit Isolation Forest on genuine data
m_model = MouseModel(contamination=0.05, n_estimators=150).fit(gen_m.values)

gen_m_scores = [m_model.score(x) for x in gen_m.values]
imp_m_scores = [m_model.score(x) for x in imp_m.values]

print(f"Genuine Mouse Mean Similarity:  {np.mean(gen_m_scores):.3f} (std: {np.std(gen_m_scores):.3f})")
print(f"Impostor Mouse Mean Similarity: {np.mean(imp_m_scores):.3f} (std: {np.std(imp_m_scores):.3f})")